# 01 — Exploratory Data Analysis

Queries Athena mart tables directly. Analyses demand distribution, day-of-week patterns, stockout frequency, and supplier reliability.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from ml.athena_client import run_query

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Load data from Athena

In [ ]:
sales = run_query("""
    SELECT sale_date, store_id, product_id, region, store_type, category,
           CAST(total_quantity_sold AS DOUBLE) AS qty,
           CAST(avg_unit_price AS DOUBLE) AS price,
           CAST(total_discount_amount AS DOUBLE) AS discount,
           CAST(day_of_week AS INTEGER) AS dow,
           CAST(is_weekend AS BOOLEAN) AS is_weekend
    FROM retailops.fct_daily_sales
""", 'sales')

sales['sale_date'] = pd.to_datetime(sales['sale_date'])
sales['qty']       = pd.to_numeric(sales['qty'], errors='coerce').fillna(0)
sales['price']     = pd.to_numeric(sales['price'], errors='coerce').fillna(0)
sales['discount']  = pd.to_numeric(sales['discount'], errors='coerce').fillna(0)
sales['dow']       = pd.to_numeric(sales['dow'], errors='coerce').fillna(0).astype(int)
sales['is_weekend'] = sales['is_weekend'].map({'true': True, 'false': False, 'True': True, 'False': False}).fillna(False)

print(f'Sales rows: {len(sales):,}')
print(f'Date range: {sales.sale_date.min().date()} → {sales.sale_date.max().date()}')
print(f'Stores: {sales.store_id.nunique()}  |  Products: {sales.product_id.nunique()}')
sales.head(3)

## 2. Demand distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# overall
axes[0].hist(sales['qty'], bins=50, edgecolor='white', color='steelblue')
axes[0].set_title('Overall demand distribution')
axes[0].set_xlabel('Units sold per store-product-day')

# by category
for cat, grp in sales.groupby('category'):
    axes[1].hist(grp['qty'], bins=30, alpha=0.5, label=cat)
axes[1].set_title('Demand by category')
axes[1].legend(fontsize=8)

# log scale
axes[2].hist(sales['qty'].clip(lower=0.1), bins=50, edgecolor='white', color='coral')
axes[2].set_xscale('log')
axes[2].set_title('Demand distribution (log scale)')

plt.tight_layout()
plt.show()

print('\nDemand summary by category:')
print(sales.groupby('category')['qty'].describe().round(2))

## 3. Day-of-week patterns

In [ ]:
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_demand = sales.groupby('dow')['qty'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(7), dow_demand.values, color='steelblue', edgecolor='white')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(dow_labels)
axes[0].set_title('Mean demand by day of week')
axes[0].set_ylabel('Mean units sold')

# by category
for cat, grp in sales.groupby('category'):
    axes[1].plot(range(7), grp.groupby('dow')['qty'].mean().values, marker='o', label=cat)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_labels)
axes[1].set_title('Day-of-week pattern by category')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# weekend vs weekday
print('\nWeekend vs weekday demand:')
print(sales.groupby('is_weekend')['qty'].agg(['mean', 'median', 'std']).round(3))

## 4. Demand trend over time

In [ ]:
daily_total = sales.groupby('sale_date')['qty'].sum().reset_index()
daily_total['rolling_7d'] = daily_total['qty'].rolling(7).mean()

plt.figure(figsize=(14, 4))
plt.plot(daily_total['sale_date'], daily_total['qty'], alpha=0.3, color='steelblue', label='Daily total')
plt.plot(daily_total['sale_date'], daily_total['rolling_7d'], color='steelblue', lw=2, label='7-day rolling mean')
plt.title('Total daily demand across all stores and products')
plt.xlabel('Date')
plt.ylabel('Total units sold')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Discount effect on demand

In [ ]:
sales['has_discount'] = (sales['discount'] > 0).astype(int)

print('Demand with vs without discount:')
print(sales.groupby('has_discount')['qty'].agg(['mean', 'median', 'count']).round(3))

# discount depth vs demand lift
disc = sales[sales['has_discount'] == 1].copy()
disc['discount_pct'] = (disc['discount'] / (disc['price'] * disc['qty']).replace(0, np.nan)).clip(0, 1)
disc['disc_bucket'] = pd.cut(disc['discount_pct'], bins=[0, 0.05, 0.1, 0.2, 0.5, 1.0])

plt.figure(figsize=(8, 4))
disc.groupby('disc_bucket')['qty'].mean().plot(kind='bar', color='coral', edgecolor='white')
plt.title('Mean demand by discount depth')
plt.xlabel('Discount %')
plt.ylabel('Mean units sold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Stockout frequency

In [ ]:
inv = run_query("""
    SELECT store_id, product_id, category,
           CAST(is_out_of_stock AS BOOLEAN) AS oos
    FROM retailops.fct_inventory_snapshots
""", 'inventory')

inv['oos'] = inv['oos'].map({'true': True, 'false': False, 'True': True, 'False': False}).fillna(False)

oos_rate = inv.groupby('category')['oos'].mean().sort_values(ascending=False)
print('Stockout rate by category:')
print(oos_rate.round(4))

plt.figure(figsize=(8, 4))
oos_rate.plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Stockout frequency by product category')
plt.ylabel('Fraction of snapshots out of stock')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Key insights summary

In [ ]:
print('=== KEY EDA FINDINGS ===')
print()
print('1. Demand distribution: right-skewed. Groceries have highest volume,')
print('   Electronics have highest unit price but lower volume.')
print()
print('2. Day-of-week pattern: visible in the data (baked into the generator).')
print('   Weekend demand differs from weekday — day_of_week is a strong feature.')
print()
print('3. No real seasonality: the generator uses fixed multipliers, so there')
print('   is no month-over-month trend. The model will learn day-of-week and')
print('   discount effects but not seasonal patterns.')
print()
print('4. Discount effect: discounted products show higher demand on average.')
print('   has_discount and discount_pct are informative features.')
print()
print('5. Stockout frequency varies by category. Stockout suppresses observed')
print('   demand — stockout_freq_14d is included as a feature to account for this.')